# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Softwareguy786/ML_INTERNSHIP_01/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*I chose logistic regression because my target is binary, a page is either declining or not. It fits the Refresh lane because it outputs a probability, which lets me rank pages into a refresh queue instead of just yes/no.*

In [2]:
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('/content/content_refresh_anonymized.csv')
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)

print(df['is_declining'].value_counts())
print("LogisticRegression loaded")

is_declining
1    16262
0    13738
Name: count, dtype: int64
LogisticRegression loaded


## 2. Split design

*I'll use a simple random 80/20 train-test split with a fixed random state, because my data has no time column and no client ID to group by. This is honest for my question: I'm not predicting the future, just scoring which pages look like declining ones.*

In [3]:
from sklearn.model_selection import train_test_split

# Features: only current-snapshot columns, no trend columns (leakage)
features = [
    'content_age_days',
    'days_since_last_update',
    'impressions_90d',
    'clicks_90d',
    'word_count',
    'ctr',
]

X = df[features].fillna(0)
y = df['is_declining']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train declining %:", round(y_train.mean()*100, 1))
print("Test declining %:", round(y_test.mean()*100, 1))

Train: (24000, 6) Test: (6000, 6)
Train declining %: 54.2
Test declining %: 54.2


## 3. Train + compare vs my baseline

*My Week-4 baseline was a simple rule: flag a page as declining if days_since_last_update > 180. I'll compare it to logistic regression on the same 80/20 split, using the same metric table (precision, recall, F1) so the comparison is honest.*

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# --- Baseline rule: stale > 180 days ---
baseline_pred = (X_test['days_since_last_update'] > 180).astype(int)

# --- Logistic regression ---
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
model_pred = model.predict(X_test)

# --- Compare ---
results = pd.DataFrame({
    'Model': ['Baseline (stale > 180d)', 'Logistic Regression'],
    'Accuracy':  [accuracy_score(y_test, baseline_pred), accuracy_score(y_test, model_pred)],
    'Precision': [precision_score(y_test, baseline_pred), precision_score(y_test, model_pred)],
    'Recall':    [recall_score(y_test, baseline_pred),    recall_score(y_test, model_pred)],
    'F1':        [f1_score(y_test, baseline_pred),        f1_score(y_test, model_pred)],
}).round(3)

print(results.to_string(index=False))

                  Model  Accuracy  Precision  Recall    F1
Baseline (stale > 180d)     0.457      0.395   0.005 0.009
    Logistic Regression     0.609      0.611   0.767 0.680


## 4. Errors and interpretation

*The baseline rule barely fired at all — recall 0.005 — so it caught almost no declining pages. Logistic regression reaches F1 = 0.68, a real improvement. But it over-flags: 1,586 healthy pages were wrongly marked as declining, against 758 declining pages missed. The strongest weight is ctr, followed by days_since_last_update. Because these features sit on very different scales, coefficient size is only a rough guide to influence. This model is decision-support, not a verdict — a human should review the ranked queue.*

In [5]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Confusion matrix
cm = confusion_matrix(y_test, model_pred, labels=[0, 1])
cm_df = pd.DataFrame(cm,
                     index=['Actual: not declining', 'Actual: declining'],
                     columns=['Predicted: not declining', 'Predicted: declining'])
print("Confusion matrix (Logistic Regression):")
print(cm_df)

# What the model leans on
coefs = pd.DataFrame({
    'feature': features,
    'coefficient': model.coef_[0]
}).sort_values('coefficient', key=abs, ascending=False).round(3)

print("\nFeature weights (bigger = more influence):")
print(coefs.to_string(index=False))

Confusion matrix (Logistic Regression):
                       Predicted: not declining  Predicted: declining
Actual: not declining                      1162                  1586
Actual: declining                           758                  2494

Feature weights (bigger = more influence):
               feature  coefficient
                   ctr       -0.050
days_since_last_update        0.004
      content_age_days       -0.002
            clicks_90d       -0.002
            word_count        0.000
       impressions_90d        0.000


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.